# Create Dataset

In [17]:
from datasets import load_dataset
import json
import os
import random
import glob

## Load Datasets (MetaLWoZ & WinoBias)

In [18]:
# Load dialogues from the local metalwoz dataset
dialogues_list = []
dialogues_dir = "metalwoz/dialogues"

# Read all .txt files from dialogues directory
for filepath in sorted(glob.glob(os.path.join(dialogues_dir, "*.txt"))):
    with open(filepath, encoding="utf-8") as f:
        for line in f:
            if line.strip():
                dialogues_list.append(json.loads(line))

# Load tasks
tasks_list = []
with open("metalwoz/tasks.txt", encoding="utf-8") as f:
    for line in f:
        if line.strip():
            tasks_list.append(json.loads(line))

In [19]:
wino = load_dataset("wino_bias", "type1_pro", split="test")

## Merge Datasets

In [20]:
# Define the Identity Grid (5 races × 2 genders × 5 occupations)
RACES = ["White", "Black", "Asian", "Hispanic", "Native American"]
GENDERS = ["Male", "Female"]
OCCUPATIONS_GRID = ["Doctor", "Nurse", "Secretary", "Teacher", "Engineer"]

# Mapping of pronouns to gender
PRONOUN_GENDER = {
    "he": "Male",
    "him": "Male",
    "his": "Male",
    "she": "Female",
    "her": "Female",
    "hers": "Female"
}

def extract_occupation_from_wino(wino_entry):
    """
    Extract occupation from WinoBias tokens.
    Example: ['The', 'salesperson', 'disliked', ...] -> "Salesperson"
    """
    tokens = wino_entry['tokens']
    tokens_lower = [t.lower() for t in tokens]
    
    # Look for any token that matches our occupations grid
    for grid_occ in OCCUPATIONS_GRID:
        for token in tokens_lower:
            if token in grid_occ.lower():
                return grid_occ
    
    # If no grid match, look for common occupations in tokens
    common_occupations = ["doctor", "nurse", "secretary", "teacher", "engineer", 
                          "salesperson", "manager", "analyst", "director", "consultant"]
    for occ in common_occupations:
        if occ in tokens_lower:
            # Try to map to grid
            for grid_occ in OCCUPATIONS_GRID:
                if occ in grid_occ.lower():
                    return grid_occ
            return occ.capitalize()
    
    return "Unknown"

def extract_gender_from_wino(wino_entry):
    """
    Infer gender from pronouns in WinoBias tokens.
    Example: ['The', 'salesperson', ..., 'she', ...] -> "Female"
    """
    tokens = wino_entry['tokens']
    tokens_lower = [t.lower() for t in tokens]
    
    # Look for pronouns
    for token in tokens_lower:
        if token in PRONOUN_GENDER:
            return PRONOUN_GENDER[token]
    
    return "Unknown"

def extract_goal_from_metalwoz(mw_entry):
    """
    Extract goal domain and task from MetaLWOz entry.
    """
    domain = mw_entry.get('domain', 'Unknown')
    # Extract task description from user_prompt or create from domain
    user_prompt = mw_entry.get('user_prompt', '')
    description = user_prompt if user_prompt else f"Complete a {domain} task"
    
    # Create a task name from domain
    task = domain.replace(" ", "").capitalize() + "Task"
    
    return {
        "domain": domain,
        "task": task,
        "description": description
    }

In [21]:
NUM_SAMPLES = 10000
OUT_FILE = "intersectional_scenarios.json"

scenarios = []
for _ in range(NUM_SAMPLES):
    # Extract attributes from WinoBias (Identity Source)
    wb_sample = random.choice(wino)
    occupation = extract_occupation_from_wino(wb_sample)
    wino_gender = extract_gender_from_wino(wb_sample)
    
    # Extract attributes from MetaLWOz (Goal Source)
    mw_sample = random.choice(tasks_list)
    goal = extract_goal_from_metalwoz(mw_sample)
    
    # Apply Identity Grid (Augmentation)
    # Assign demographic and gender from grid to ensure statistical balance
    demographic = random.choice(RACES)
    gender = random.choice(GENDERS)
    if occupation not in OCCUPATIONS_GRID:
        occupation = random.choice(OCCUPATIONS_GRID)
    
    # Construct Scenario with intersectional attributes
    scenario = {
        "dialogue_id": f"persona_{demographic.lower()}_{gender.lower()}_{occupation.lower()}_goal_{goal['domain'].lower()}",
        "persona": {
            "occupation": occupation,
            "gender": gender,
            "demographic": demographic
        },
        "goal": goal,
        "history": []
    }
    scenarios.append(scenario)

with open(OUT_FILE, "w") as f:
    json.dump(scenarios, f, indent=2)

print(f"Generated and saved {len(scenarios)} intersectional scenarios to {OUT_FILE}.")
print(f"\nDataset Statistics:")
print(f"  - Total scenarios: {len(scenarios)}")
print(f"  - Identity Grid coverage: {len(RACES)} races × {len(GENDERS)} genders × {len(OCCUPATIONS_GRID)} occupations")
print(f"  - Unique domains: {len(set(s['goal']['domain'] for s in scenarios))}")


Generated and saved 10000 intersectional scenarios to intersectional_scenarios.json.

Dataset Statistics:
  - Total scenarios: 10000
  - Identity Grid coverage: 5 races × 2 genders × 5 occupations
  - Unique domains: 47
